# NYISO Solar Forecasting: Model Training

## Imports and Configuration

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.statespace.sarimax import SARIMAX
HAS_LIGHTGBM = True
try:
    from lightgbm import LGBMRegressor
except Exception:
    HAS_LIGHTGBM = False

HAS_XGBOOST = True
try:
    from xgboost import XGBRegressor
except Exception:
    HAS_XGBOOST = False

HAS_AUTOGLUON = True
try:
    from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
except Exception:
    HAS_AUTOGLUON = False

HAS_TFT = True
try:
    import torch
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
    from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
    from pytorch_forecasting.data import GroupNormalizer
    from pytorch_forecasting.metrics import MAE
except Exception:
    HAS_TFT = False

HAS_TIMEGPT = True
try:
    from nixtla import NixtlaClient
except Exception:
    HAS_TIMEGPT = False

HAS_GRANITE_TTM = True
try:
    from tsfm_public import TinyTimeMixerForPrediction
except Exception:
    HAS_GRANITE_TTM = False

In [3]:
repo_root = Path.home() / "Documents" / "Coding" / "ML_NYISOSolarForecast"

data_root = repo_root / "data"
processed_dir = data_root / "processed"

model_ready_in = processed_dir / "04_system_model_ready_data.csv"

split_date = pd.Timestamp("2024-07-01 00:00:00+00:00")
validation_start = pd.Timestamp("2024-01-01 00:00:00+00:00")

In [4]:
df_model = pd.read_csv(model_ready_in, low_memory=False)

df_model.columns = (
    df_model.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

df_model["time_stamp"] = pd.to_datetime(df_model["time_stamp"], utc=True, errors="coerce")
df_model["time_local"] = df_model["time_stamp"].dt.tz_convert("America/New_York")

print("Shape:", df_model.shape)
print("Time Range:", df_model["time_stamp"].min(), "to", df_model["time_stamp"].max())
print("Columns:")
print(df_model.columns.tolist())

Shape: (41455, 30)
Time Range: 2020-11-17 05:00:00+00:00 to 2025-09-19 03:00:00+00:00
Columns:
['time_stamp', 'time_local', 'zone_name', 'dataset_split', 'actual_mw', 'forecast_mw', 'forecast_error_mw', 'temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'shortwave_ramp_abs', 'is_morning_ramp', 'is_midday']


## Define Target, Features, and Chronological Splits

In [7]:
target = "forecast_error_mw"

required_cols = [
    "time_stamp",
    "time_local",
    "zone_name",
    "dataset_split",
    "actual_mw",
    "forecast_mw",
    "forecast_error_mw",
]

missing_required = [
    c for c in required_cols 
    if c not in df_model.columns
]

if missing_required:
    raise ValueError(f"Missing Necessary Columns in Dataset: {missing_required}")


feature_cols = [
    c for c in df_model.columns
    if c not in required_cols
]

print("\nTarget:", target)
print("Number of Features:", len(feature_cols))
print("Feature Columns:")
print(feature_cols)


Target: forecast_error_mw
Number of Features: 23
Feature Columns:
['temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'shortwave_ramp_abs', 'is_morning_ramp', 'is_midday']


In [8]:
X = df_model[feature_cols].copy()
y = df_model[target].copy()

train_mask = (
    df_model["dataset_split"].eq("train")
    & y.notna()
)

test_mask = (
    df_model["dataset_split"].eq("test")
    & y.notna()
    & df_model["actual_mw"].notna()
    & df_model["forecast_mw"].notna()
)

subtrain_mask = (
    df_model["dataset_split"].eq("train")
    & df_model["time_stamp"].lt(validation_start)
    & y.notna()
)

valid_mask = (
    df_model["dataset_split"].eq("train")
    & df_model["time_stamp"].ge(validation_start)
    & y.notna()
    & df_model["actual_mw"].notna()
    & df_model["forecast_mw"].notna()
)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()
X_subtrain = X.loc[subtrain_mask].copy()
X_valid = X.loc[valid_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()
y_subtrain = y.loc[subtrain_mask].copy()
y_valid = y.loc[valid_mask].copy()

baseline_actual_test = df_model.loc[test_mask, "actual_mw"].copy()
baseline_forecast_test = df_model.loc[test_mask, "forecast_mw"].copy()

baseline_actual_valid = df_model.loc[valid_mask, "actual_mw"].copy()
baseline_forecast_valid = df_model.loc[valid_mask, "forecast_mw"].copy()

daylight_test_mask = df_model.loc[test_mask, "shortwave_radiation"] > 0
daylight_valid_mask = df_model.loc[valid_mask, "shortwave_radiation"] > 0

In [10]:
print("\nTarget:", target)
print("Number of features:", len(feature_cols))
print("Feature columns:")
print(feature_cols)

print("\nTrain/Test/Validation Shapes")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_subtrain:", X_subtrain.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("y_subtrain:", y_subtrain.shape)
print("y_valid:", y_valid.shape)


Target: forecast_error_mw
Number of features: 23
Feature columns:
['temperature_2m', 'surface_pressure', 'cloud_cover', 'windspeed_10m', 'shortwave_radiation', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'dayofyear_sin', 'forecast_x_hour_sin', 'forecast_x_hour_cos', 'shortwave_x_cloud', 'shortwave_x_temp', 'forecast_roll_mean_3', 'shortwave_roll_mean_3', 'forecast_roll_mean_24', 'shortwave_roll_mean_24', 'forecast_diff_1', 'shortwave_diff_1', 'shortwave_ramp_abs', 'is_morning_ramp', 'is_midday']

Train/Test/Validation Shapes
X_train: (30921, 23)
X_test: (10534, 23)
X_subtrain: (26630, 23)
X_valid: (4291, 23)
y_train: (30921,)
y_test: (10534,)
y_subtrain: (26630,)
y_valid: (4291,)


In [11]:
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert X_subtrain.shape[0] == y_subtrain.shape[0]
assert X_valid.shape[0] == y_valid.shape[0]

In [12]:
feature_diagnostics = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [str(X_train[c].dtype) for c in feature_cols],
    "missing_train": [int(X_train[c].isna().sum()) for c in feature_cols],
    "missing_train_pct": [100 * X_train[c].isna().mean() for c in feature_cols],
    "nunique_train": [X_train[c].nunique(dropna=True) for c in feature_cols],
    "std_train": [X_train[c].std(skipna=True) for c in feature_cols],
}).sort_values(
    ["missing_train_pct", "nunique_train", "feature"],
    ascending=[False, True, True]
).reset_index(drop=True)

print("\nFrozen Feature Diagnostics")
print(feature_diagnostics)


Frozen Feature Diagnostics
                   feature    dtype  missing_train  missing_train_pct  \
0    forecast_roll_mean_24  float64             11           0.035575   
1          forecast_diff_1  float64             11           0.035575   
2     forecast_roll_mean_3  float64             11           0.035575   
3       shortwave_ramp_abs  float64              1           0.003234   
4         shortwave_diff_1  float64              1           0.003234   
5    shortwave_roll_mean_3  float64              1           0.003234   
6   shortwave_roll_mean_24  float64              1           0.003234   
7                is_midday    int64              0           0.000000   
8          is_morning_ramp    int64              0           0.000000   
9                month_cos  float64              0           0.000000   
10               month_sin  float64              0           0.000000   
11                hour_sin  float64              0           0.000000   
12                hour_

In [13]:
corr_input = X_train.copy()
corr_input = corr_input.fillna(corr_input.median(numeric_only=True))

corr_matrix = corr_input.corr(numeric_only=True).abs()

high_corr_pairs = []
cols = corr_matrix.columns.tolist()

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        corr_val = corr_matrix.iloc[i, j]
        if corr_val >= 0.98:
            high_corr_pairs.append((cols[i], cols[j], corr_val))

high_corr_pairs_df = pd.DataFrame(
    high_corr_pairs,
    columns=["feature_1", "feature_2", "abs_corr"]
).sort_values("abs_corr", ascending=False)

print("\nHigh-Correlation Pairs (|corr| >= 0.98)")
print(high_corr_pairs_df if len(high_corr_pairs_df) > 0 else "nothing found")


High-Correlation Pairs (|corr| >= 0.98)
nothing found
